In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ============================================================
#                     PARAMETERS
# ============================================================
m, n, b, M = 5, 50, 50, 20       # synthetic setting
eta_OV = 1.0                         # fix eta_OV and sweep eta_QK
ratios = [0.01, 0.1, 1, 10, 100]      # eta_QK / eta_OV

# baseline horizon guess (auto-extend if needed)
T_tau_init = 2e5

dt_tau = 10                        # RK4 step in tau (smaller => more accurate, slower)

EXP_CLIP = 60
EPS = 1e-12

# target: make sure we run until each ratio hits this loss
TARGET_LOSS = 1e-4
MAX_T_TAU = 5e7                      # hard cap to avoid infinite runs

# Stopping-loss thresholds requested (linear x-axis)

L_max =1
L_min = 1e-4
stop_thresholds = np.logspace(np.log10(L_max), np.log10(L_min), 50)

# Plot styling
LABEL_FONTSIZE = 172
TICK_FONTSIZE = 130
LEGEND_FONTSIZE = 140
LINEWIDTH = 40

# ============================================================
#                     UTILITIES
# ============================================================
def safe_exp(x):
    """Clipped exponential for numerical stability."""
    return np.exp(np.clip(x, -EXP_CLIP, EXP_CLIP))

# ============================================================
#              ODEs (original time t)
# ============================================================
def alpha_of_mu_QK(mu_QK):
    """
    alpha(t) = (m e^{mu_QK}) / (m e^{mu_QK} + n)
    """
    exp_mu = safe_exp(mu_QK)
    return (m * exp_mu) / (m * exp_mu + n + EPS)

def dmu_OV_dt(mu_OV, mu_QK, eta_OV_):
    """
    d mu_OV / dt = (eta_OV * alpha) / ( b ( exp(alpha*mu_OV) + M - 1 ) )
    """
    alpha = alpha_of_mu_QK(mu_QK)
    denom = b * (safe_exp(alpha * mu_OV) + M - 1) + EPS
    return (eta_OV_ * alpha) / denom

def dmu_QK_dt(mu_OV, mu_QK, eta_QK_):
    """
    d mu_QK / dt = eta_QK * (M-1) * alpha(1-alpha) * mu_OV /
                   ( M b^2 ( exp(alpha*mu_OV) + M - 1 ) )
    """
    alpha = alpha_of_mu_QK(mu_QK)
    denom = (b * b) * M * (safe_exp(alpha * mu_OV) + M - 1) + EPS
    num = eta_QK_ * (M - 1) * alpha * (1 - alpha) * mu_OV
    return num / denom

# ============================================================
#                 LOSS FUNCTION (correct class)
# ============================================================
def stable_loss(mu_OV, mu_QK):
    """
    loss = log( 1 + (M-1) exp(-mu_OV * alpha) )
    """
    alpha = alpha_of_mu_QK(mu_QK)
    logit = mu_OV * alpha
    return np.log1p((M - 1) * safe_exp(-logit))

# ============================================================
#          RK4 in normalized time tau = t * min_eta
# ============================================================
def rhs_tau(mu_OV, mu_QK, eta_OV_, eta_QK_, min_eta):
    """
    Integrate in tau: d/dtau = (1/min_eta) d/dt
    """
    f1 = dmu_OV_dt(mu_OV, mu_QK, eta_OV_) / min_eta
    f2 = dmu_QK_dt(mu_OV, mu_QK, eta_QK_) / min_eta
    return np.array([f1, f2], dtype=float)

def rk4_step(mu_OV, mu_QK, eta_OV_, eta_QK_, min_eta, dtau):
    y0 = np.array([mu_OV, mu_QK], dtype=float)

    k1 = rhs_tau(y0[0], y0[1], eta_OV_, eta_QK_, min_eta)
    k2 = rhs_tau(y0[0] + 0.5*dtau*k1[0], y0[1] + 0.5*dtau*k1[1], eta_OV_, eta_QK_, min_eta)
    k3 = rhs_tau(y0[0] + 0.5*dtau*k2[0], y0[1] + 0.5*dtau*k2[1], eta_OV_, eta_QK_, min_eta)
    k4 = rhs_tau(y0[0] + dtau*k3[0], y0[1] + dtau*k3[1], eta_OV_, eta_QK_, min_eta)

    y1 = y0 + (dtau/6.0) * (k1 + 2*k2 + 2*k3 + k4)

    # Theory keeps these >= 0; enforce for stability.
    y1[0] = max(float(y1[0]), 0.0)
    y1[1] = max(float(y1[1]), 0.0)

    # Mild clipping (rarely needed; helps if dt_tau is too large)
    y1[0] = min(y1[0], 1e6)
    y1[1] = min(y1[1], 1e6)
    return y1[0], y1[1]

# ============================================================
#        SIMULATION ROUTINE (auto-extend until target loss)
# ============================================================
def run_simulation_until(eta_OV_, eta_QK_, init_mu_OV=0.0, init_mu_QK=0.0,
                         target_loss=1e-5, T_tau_init=2e5, max_T_tau=5e7):
    """
    Integrate in tau until loss <= target_loss, auto-extending horizon if needed.
    Hard-stops at max_T_tau.
    """
    min_eta = min(eta_OV_, eta_QK_)

    # storage as Python lists for easy extension
    tau_list = [0.0]
    mu_OV_list = [float(init_mu_OV)]
    mu_QK_list = [float(init_mu_QK)]

    alpha0 = alpha_of_mu_QK(mu_QK_list[0])
    logit0 = mu_OV_list[0] * alpha0
    loss0 = stable_loss(mu_OV_list[0], mu_QK_list[0])

    alpha_list = [float(alpha0)]
    logit_list = [float(logit0)]
    loss_list  = [float(loss0)]

    # horizon in tau for this run (will grow)
    T_tau_current = float(T_tau_init)

    def step_once():
        muov, muqk = rk4_step(mu_OV_list[-1], mu_QK_list[-1], eta_OV_, eta_QK_, min_eta, dt_tau)
        tau_new = tau_list[-1] + dt_tau

        tau_list.append(tau_new)
        mu_OV_list.append(muov)
        mu_QK_list.append(muqk)

        a = alpha_of_mu_QK(muqk)
        lgt = muov * a
        ls = stable_loss(muov, muqk)

        alpha_list.append(float(a))
        logit_list.append(float(lgt))
        loss_list.append(float(ls))

    # Run in chunks; if not reached, extend
    while True:
        while tau_list[-1] < T_tau_current:
            step_once()
            if loss_list[-1] <= target_loss:
                break

        if loss_list[-1] <= target_loss:
            break

        if T_tau_current >= max_T_tau:
            print(f"[WARN] η_QK/η_OV={eta_QK_/eta_OV_:.3g}: "
                  f"did NOT reach loss <= {target_loss} by tau={T_tau_current:.3g}. "
                  f"Final loss={loss_list[-1]:.3e}")
            break

        T_tau_current = min(2.0 * T_tau_current, max_T_tau)

    # Convert to arrays
    tau = np.array(tau_list, dtype=float)
    mu_OV = np.array(mu_OV_list, dtype=float)
    mu_QK = np.array(mu_QK_list, dtype=float)
    alpha_t = np.array(alpha_list, dtype=float)
    logit = np.array(logit_list, dtype=float)
    loss = np.array(loss_list, dtype=float)

    mid = len(tau) // 2
    print(f"η_QK/η_OV={eta_QK_/eta_OV_:.2g}: "
          f"tau_end={tau[-1]:.3g}, "
          f"mu_OV {mu_OV[mid]:.3f}->{mu_OV[-1]:.3f}, "
          f"mu_QK {mu_QK[mid]:.3f}->{mu_QK[-1]:.3f}, "
          f"alpha {alpha_t[mid]:.3f}->{alpha_t[-1]:.3f}, "
          f"loss {loss[mid]:.3e}->{loss[-1]:.3e}")

    return tau, mu_OV, mu_QK, loss, alpha_t, logit

# ============================================================
#                     RUN SWEEP (auto-extend)
# ============================================================
eta_pairs = [(eta_OV, eta_OV * r) for r in ratios]
results = {}
for eov, eqk in eta_pairs:
    results[(eov, eqk)] = run_simulation_until(
        eov, eqk,
        target_loss=TARGET_LOSS,
        T_tau_init=T_tau_init,
        max_T_tau=MAX_T_TAU
    )

# ============================================================
#                        PLOTTING HELPERS
# ============================================================
font_properties = fm.FontProperties(weight='bold', size=LEGEND_FONTSIZE)





# ============================================================
#                ORIGINAL TIME-SERIES PLOTS (optional)
# ============================================================


# # ============================================================
# #                NEW STOPPING-LOSS PLOTS (requested)
# # ============================================================
# plot_stopping_curve_linear_x(r"$\bf\mu_{OV}\ \text{at stop}$",
#                              lambda muov_at, muqk_at, alpha_at: muov_at,
#                              "mu_OV_vs_stopping_loss_linearx.pdf")

# plot_stopping_curve_linear_x(r"$\bf\mu_{QK}\ \text{at stop}$",
#                              lambda muov_at, muqk_at, alpha_at: muqk_at,
#                              "mu_QK_vs_stopping_loss_linearx.pdf")

# plot_stopping_curve_linear_x(r"$\bf\alpha(\tau)\ \text{at stop}$",
#                              lambda muov_at, muqk_at, alpha_at: alpha_at,
#                              "alpha_vs_stopping_loss_linearx.pdf")

# print("Done.")
# print(f"Params: m={m}, n={n}, b={b}, M={M}, eta_OV={eta_OV}, ratios={ratios}")
# print(f"Stopping thresholds: {stop_thresholds}")


η_QK/η_OV=0.01: tau_end=2.55e+06, mu_OV 78.797->81.669, mu_QK 0.521->0.559, alpha 0.144->0.149, loss 2.215e-04->1.000e-04
η_QK/η_OV=0.1: tau_end=4.9e+06, mu_OV 41.216->42.282, mu_QK 1.334->1.395, alpha 0.275->0.287, loss 2.253e-04->1.000e-04


In [ ]:
def plot_quantity_log1p(ylabel, extractor, filename, T_tau_for_ticks=None):
    """
    Plot y(tau) vs x=log10(1+tau) (your original style),
    but using the actual simulated tau arrays.
    """
    fig, ax = plt.subplots(figsize=(48, 48))

    # If not provided, set tick range from max tau across all curves
    if T_tau_for_ticks is None:
        T_tau_for_ticks = max(results[(eov, eqk)][0][-1] for (eov, eqk) in eta_pairs)

    for (eov, eqk) in eta_pairs:
        tau, mu_OV, mu_QK, loss, alpha_t, logit = results[(eov, eqk)]
        y = extractor(mu_OV, mu_QK, loss, alpha_t, logit)
        x = np.log10(1.0 + tau)
        ax.plot(x, y, linewidth=LINEWIDTH,
                 label=rf'$\bf r={eqk/eov:.3g}$',solid_capstyle="round")
    ax.set_xlabel(r"Normalized time $t$", fontweight="bold", fontsize=172, labelpad=24)
    ax.set_ylabel(ylabel,fontweight="bold", fontsize=172, labelpad=22)
    
    
    max_power = int(np.floor(np.log10(max(1.0, T_tau_for_ticks))))
    raw_ticks_tau = [0] + [10**k for k in range(0, max_power + 1)]
    tick_pos = [np.log10(1.0 + t) for t in raw_ticks_tau]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels([r"$0$", r"$1$", r"$10$",r"$10^2$", r"$10^3$", r"$10^4$", r"$10^5$" ,r"$10^6$"])
    
    #ax.set_yticks([0, 20, 40, 60,80])
    
    # Tick label sizes (as you had) + tick line thickness/length + padding
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=130,
        width=24,        # tick thickness
        length=36,      # tick length
        direction="out",
        pad=18         # push tick labels away from axes (reduces overlap with plot)
    )
     # Make tick labels bold (tick_params doesn't set fontweight reliably everywhere)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")   
    # Thicken the axis box (spines) to match thick ticks/lines
    for spine in ax.spines.values():
        spine.set_linewidth(10)# Add a little internal margin so curves don't hug the axes/tick labels
    ax.margins(x=0.02, y=0.08)
    # Optional: if you re-enable legend, keep it out of the data
    ax.legend(frameon=True, fontsize=140, loc="upper left", bbox_to_anchor=(0.02, 0.98))
        
    fig.tight_layout(pad=0.4)

    plt.savefig(filename, bbox_inches='tight')

In [ ]:
plot_quantity_log1p(r"$\bf\mu_{OV}(t)$",
                    lambda mu_OV, mu_QK, loss, alpha_t, logit: mu_OV,
                    "mu_OV_1.pdf")

plot_quantity_log1p(r"$\bf\mu_{QK}(t)$",
                    lambda mu_OV, mu_QK, loss, alpha_t, logit: mu_QK,
                    "mu_QK_1.pdf")

plot_quantity_log1p(r"Loss",
                    lambda mu_OV, mu_QK, loss, alpha_t, logit: loss,
                    "loss_t_1.pdf")



In [ ]:
plot_quantity_log1p(r"$\alpha(\tau)$",
                    lambda mu_OV, mu_QK, loss, alpha_t, logit: alpha_t,
                    "alpha_t_1.pdf")

In [ ]:
def stopping_value_vs_threshold(mu_OV, mu_QK, loss, alpha_t, thresholds):
    """
    For each threshold L, find the earliest index i where loss[i] <= L.
    Return arrays of mu_OV[i], mu_QK[i], alpha_t[i] at that stopping index.
    If never reached, returns NaN.
    """
    muov_at = np.full(thresholds.shape, np.nan, dtype=float)
    muqk_at = np.full(thresholds.shape, np.nan, dtype=float)
    alpha_at = np.full(thresholds.shape, np.nan, dtype=float)

    for j, L in enumerate(thresholds):
        idx = np.where(loss <= L)[0]
        if idx.size > 0:
            i0 = int(idx[0])
            muov_at[j] = mu_OV[i0]
            muqk_at[j] = mu_QK[i0]
            alpha_at[j] = alpha_t[i0]

    return muov_at, muqk_at, alpha_at

In [ ]:
def plot_stopping_curve_linear_x(ylabel, y_selector, filename):
    fig, ax = plt.subplots(figsize=(48, 48))
    
    

    for (eov, eqk) in eta_pairs:
        tau, mu_OV, mu_QK, loss, alpha_t, logit = results[(eov, eqk)]
        muov_at, muqk_at, alpha_at = stopping_value_vs_threshold(
            mu_OV, mu_QK, loss, alpha_t, stop_thresholds
        )
        y = y_selector(muov_at, muqk_at, alpha_at)

        plt.plot(
            stop_thresholds, y,
            linewidth=LINEWIDTH,
            marker='o',
            label=rf'$\bfr={eqk/eov:.3g}$'
        )

    ax.set_xscale("log")
    ax.invert_xaxis()

    # Linear x-axis (as requested). Show decreasing thresholds left->right.
    #ax.invert_xaxis()

    # Force exact tick positions/labels you requested
    
    ax.set_xlabel(r"Stopping loss threshold", fontweight="bold", fontsize=172, labelpad=24)
    ax.set_ylabel(ylabel,fontweight="bold", fontsize=172, labelpad=22)
    
    
    # Tick label sizes (as you had) + tick line thickness/length + padding
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=130,
        width=24,        # tick thickness
        length=36,      # tick length
        direction="out",
        pad=18         # push tick labels away from axes (reduces overlap with plot)
    )
     # Make tick labels bold (tick_params doesn't set fontweight reliably everywhere)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")   
    # Thicken the axis box (spines) to match thick ticks/lines
    for spine in ax.spines.values():
        spine.set_linewidth(10)# Add a little internal margin so curves don't hug the axes/tick labels
    ax.margins(x=0.02, y=0.08)
    # Optional: if you re-enable legend, keep it out of the data
    ax.legend(frameon=True, fontsize=140, loc="upper left", bbox_to_anchor=(0.02, 0.98))
        
    fig.tight_layout(pad=0.4)

    plt.savefig(filename, bbox_inches='tight')

In [ ]:
# ============================================================
plot_stopping_curve_linear_x(r"$\bf{\mu_{OV}}$",
                             lambda muov_at, muqk_at, alpha_at: muov_at,
                             "mu_OV_stopping_loss_1.pdf")

plot_stopping_curve_linear_x(r"$\bf{\mu_{QK}}$",
                             lambda muov_at, muqk_at, alpha_at: muqk_at,
                             "mu_QK_stopping_loss_1.pdf")



In [ ]:
plot_stopping_curve_linear_x(r"$\bf{\alpha(\tau)}$",
                             lambda muov_at, muqk_at, alpha_at: alpha_at,
                             "alpha_stopping_loss_1.pdf")

In [ ]:
# _1 means m, n, b, M = 5, 50, 50, 20       # synthetic setting